# **HANDS - ON: FUNDAMENTOS DE LLMS CON MODELOS LLAMA**


Una vez vista la masterclass ***Fundamentos de LLMs y Arquitectura de Llama***, se proporciona el siguiente ***Colab*** para construir, en vivo, la primera llamada a Llama y medir su comportamiento.

Usamos **Groq** para tener acceso a inferencia de Llama con GPU gratuita, sin necesidad de infraestructura propia.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1bjMHw7fmMxo9dyQp140e9mqotKRkfKBg?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

Antes de llamar a Llama necesitamos activar la GPU gratuita de Colab y conectar nuestra cuenta de **Groq**. La librería `groq` es el cliente oficial en Python para hacer llamadas a la API.

### **COLAB SECRETS**

Para no exponer tu ***API key*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la ***API key*** para guardarla dentro de una variable y usarla dentro del notebook.

In [2]:
# Instalar cliente de Groq y leer API key desde Colab Secrets
!pip install groq -q

import os
from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print("Cliente de Groq inicializado correctamente.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 6.3 MB/s eta 0:00:00
Cliente de Groq inicializado correctamente.


## **DE PALABRAS A TOKENS**

Antes de que Llama genere una sola palabra, tu prompt se divide en **tokens**. Vamos a definir un prompt de ejemplo que después enviaremos a Llama.

In [3]:
# Definir un prompt de ejemplo (una pregunta real de tu propio contexto)
prompt = "Se pretende hacer la instalación de una IA para Asistir a un operador en setear la temperatura de un horno de piezas metálicas"
# prompt = "Explica en un párrafo qué hace un router doméstico"
print(prompt)


Se pretende hacer la instalación de una IA para Asistir a un operador en setear la temperatura de un horno de piezas metálicas


### **PRIMERA LLAMADA A LLAMA (ZERO-SHOT)**

Un LLM predice el siguiente token más probable, **no busca la respuesta en una base de datos**. Vamos a enviar el prompt a Llama en modo zero-shot (sin ejemplos previos) y ver la respuesta generada.

In [5]:
# Enviar el prompt a Llama en Groq y mostrar la respuesta generada
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt}]
)

### **CUÁNTOS TOKENS CONSUMIÓ EL PROMPT**

La respuesta de la API trae un resumen (`usage`) con el número de tokens de entrada y de salida. Es la forma más directa de responder: ¿cuántos tokens consumió mi prompt?

In [7]:
# Mostrar cuántos tokens tuvo el prompt y cuántos tuvo la respuesta
print("Tokens del prompt:", response.usage.prompt_tokens)
print("Tokens de la respuesta:", response.usage.completion_tokens)
print("Tokens totales:", response.usage.total_tokens)


Tokens del prompt: 98
Tokens de la respuesta: 2048
Tokens totales: 2146


### **MIDIENDO LA LATENCIA DE INFERENCIA**

Elegir el tamaño correcto de modelo es una decisión de ingeniería, no solo de potencia bruta. Vamos a medir cuánto tarda Llama en responder el mismo prompt.

In [9]:
# Medir el tiempo de respuesta de Llama para el mismo prompt
import time

inicio = time.time()
response_tiempo = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt}]
)
duracion = time.time() - inicio
# duracion_ms = duracion * 1000  # si prefieres reportarlo en milisegundos

print(f"Tiempo de respuesta: {duracion:.2f} segundos")


Tiempo de respuesta: 2.64 segundos


### **COMPARANDO DOS TAMAÑOS DE MODELO**

Repite la misma llamada usando la versión más grande de Llama disponible en Groq y compara tiempo de respuesta y calidad contra el modelo ligero.

In [11]:
# Repetir la llamada con un modelo Llama más grande y comparar tiempo y calidad
inicio = time.time()
response_grande = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": prompt}]
)
duracion_grande = time.time() - inicio

print(f"Modelo ligero: {duracion:.2f} s — {response.usage.total_tokens} tokens")
print(f"Modelo grande: {duracion_grande:.2f} s — {response_grande.usage.total_tokens} tokens")
print("\nRespuesta del modelo grande:\n", response_grande.choices[0].message.content)

Modelo ligero: 2.64 s — 2146 tokens
Modelo grande: 6.93 s — 3170 tokens

Respuesta del modelo grande:
 ¡Perfecto! Vamos a estructurar la puesta en marcha de una IA que ayude al operador a **“setear” (definir y ajustar) la temperatura de un horno de piezas metálicas**. A continuación tienes una hoja de ruta práctica, dividida en bloques funcionales, con los puntos clave que deberías cubrir, las decisiones técnicas habituales y unas recomendaciones de seguridad y gestión del proyecto.

---

## 1️⃣  Definición del alcance y requisitos

| Pregunta | Por qué es importante | Qué respuesta deberías buscar |
|----------|----------------------|------------------------------|
| **Objetivo concreto**: ¿La IA solo sugiere la temperatura o también controla el set point automáticamente? | Define el nivel de autonomía (asistente vs controlador). | “Sugerir” → UI + alerta. “Controlar” → bucle cerrado con PLC. |
| **Tipo de piezas** (material, geometría, espesor, tratamiento posterior). | Cada aleación

## **CHALLENGE: COMPARADOR DE MODELOS LLAMA**

Una vez visto el ***Hands-On: Fundamentos de LLMs***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un pequeño **comparador que envíe 3 preguntas** reales a Llama y registre, para cada una, el **tiempo de respuesta** y el **número de tokens** consumidos.

**IMPORTANTE:** Para su revisión, **es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.**

### **INSTRUCCIONES:**

**1. Carga la API key y genera 3 preguntas:**

   * Lee la API key de llama ya configurada desde **Colab Secrets**.
   * Construye una lista vacía llamada `preguntas` y agrégale 3 preguntas frecuentes de tu propio contexto (soporte técnico, tienda, escuela, etc.).

In [19]:
# Leer API key desde Colab Secrets
!pip install groq -q

import os
from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print("Cliente de Groq inicializado correctamente.")

Cliente de Groq inicializado correctamente.


In [20]:
# Definir la lista de preguntas
pregunta1 = "¿Que se tienen que hacer para instalar una IA en una computadora local?"
pregunta2 = "¿Debo actualizar periodicamente el modelo para tener un mejor resultado?"
pregunta3 = "¿Cuantos valores requiero para entrenar un modelo?"
# prompt = "Explica en un párrafo qué hace un router doméstico"
print(pregunta1)
print(pregunta2)
print(pregunta3)


¿Que se tienen que hacer para instalar una IA en una computadora local?
¿Debo actualizar periodicamente el modelo para tener un mejor resultado?
¿Cuantos valores requiero para entrenar un modelo?


**2. Consulta la primera pregunta:** Envía la primera pregunta a Llama usando el modelo más ligero disponible en Groq. Guarda la respuesta, el tiempo de respuesta y el número de tokens en un diccionario llamado `resultado_1`.

In [21]:
# Consultar la primera pregunta y guardar el resultado en resultado_1
resultado1 = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": pregunta1}]
)

print(resultado1.choices[0].message.content)


## Cómo “instalar” una IA en una computadora local

El término *instalar* puede sonar como si se tratara de un programa que se instala como un juego, pero en realidad se trata de:

1. **Preparar el entorno de hardware / software**  
2. **Instalar los frameworks y librerías necesarias**  
3. **Obtener el modelo (pre‑entrenado o entrenarlo tú mismo)**  
4. **Ejecutar el modelo (inferencia o entrenamiento)**

A continuación te dejo una guía paso‑a‑paso con los pasos más comunes, pensando en un entorno **Linux / macOS / Windows** con **Python**. Si tu objetivo es algo distinto (por ejemplo una IA que corre en un microcontrolador) la ruta cambiaría, pero la idea básica sigue siendo la misma.

---

## 1. Requisitos de hardware

| Requisito | Por qué es importante | Ejemplo |
|-----------|----------------------|---------|
| **CPU** | El mínimo imprescindible, pero muy lento si no hay GPU | Intel i5 o AMD Ryzen 5 |
| **GPU (opcional, pero recomendado)** | Acelera el entrenamiento y la inferenc

**3. Repite para la segunda y tercera pregunta:** Crea `resultado_2` y `resultado_3` de la misma forma.

In [15]:
# Consultar la segunda pregunta y guardar el resultado en resultado_2


In [22]:
# Consultar la primera pregunta y guardar el resultado en resultado_1
resultado2 = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": pregunta2}]
)

print(resultado2.choices[0].message.content)


### Sí, actualizar tu modelo suele ser una buena práctica, pero no siempre es obligatorio.  
A continuación te dejo una guía rápida para decidir cuándo y cómo hacerlo:

| **Situación** | **¿Actualizar?** | **Cómo hacerlo** | **Cuándo esperar** |
|---------------|------------------|------------------|--------------------|
| **Cambios en los datos (data drift)** | Sí | 1. Monitorea métricas de rendimiento (precisión, recall, error, etc.) en tiempo real.<br>2. Recolecta un lote reciente de datos.<br>3. Reentrena el modelo (o fine‑tune) con el conjunto nuevo.<br>4. Evalúa en un conjunto de validación separado. | Si la diferencia entre métricas actuales y de entrenamiento supera un umbral predefinido (ej. 5 % de caída). |
| **Mejoras de arquitectura/hyper‑params** | Sí | 1. Experimenta con nuevas arquitecturas o hiperparámetros en un entorno de pruebas.<br>2. Usa búsqueda automática (grid, random, Bayesian, Hyperband).<br>3. Selecciona el mejor modelo según métricas de validación.<br>4. Des

In [16]:
# Consultar la tercera pregunta y guardar el resultado en resultado_3


In [23]:
# Consultar la primera pregunta y guardar el resultado en resultado_1
resultado3 = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": pregunta3}]
)

print(resultado3.choices[0].message.content)


El número de ejemplos (o “valores”) que necesitas para entrenar un modelo depende de varios factores. A continuación tienes una guía general, que puedes adaptar según tu caso concreto:

| **Factor** | **Qué significa** | **Recomendación típica** |
|------------|-------------------|--------------------------|
| **Complejidad del modelo** | Número de parámetros que el modelo tiene que aprender. | • Modelos lineales o con pocos parámetros: **algunas centenas** de ejemplos son suficientes.<br>• Redes neuronales pequeñas (p. ej. 1‑2 capas con < 1 000 neuronas): **varios miles**.<br>• Redes profundas (CNN, RNN, transformers): **decenas o cientos de miles** (idealmente millones). |
| **Dimensión de los datos** (número de características) | Cuantas variables (features) haya por ejemplo. | Un principio de “regla de 10‑30 veces más ejemplos que parámetros” suele funcionar. Si tienes 100 features, un modelo lineal simple no necesita más de unas 1 000‑3 000 ejemplos, pero si cada feature se vuelve

**4. Junta los resultados:** Agrega `resultado_1`, `resultado_2` y `resultado_3` a una lista vacía llamada `resultados`.

In [24]:
# Definir la lista resultados y agregar los tres diccionarios
resultados = []
resultados.append(resultado1)
resultados.append(resultado2)
resultados.append(resultado3)

**5. Muestra la comparación:** Imprime `resultados` y concluye si el modelo ligero resolvió las 3 preguntas satisfactoriamente.

In [27]:
# Mostrar la tabla final de resultados
import pandas as pd

df = pd.DataFrame(resultados)

print(df)

                                                  0   \
0  (id, chatcmpl-0d8d4f53-3cbc-4935-8d4a-7f0bbcbf...   
1  (id, chatcmpl-2e8df74e-e7ea-4267-8e79-45bd1a1a...   
2  (id, chatcmpl-367cae5b-2227-43d7-b624-f3b388ce...   

                                                  1                      2   \
0  (choices, [Choice(finish_reason='length', inde...  (created, 1788551491)   
1  (choices, [Choice(finish_reason='stop', index=...  (created, 1788551572)   
2  (choices, [Choice(finish_reason='stop', index=...  (created, 1788551633)   

                            3                          4   \
0  (model, openai/gpt-oss-20b)  (object, chat.completion)   
1  (model, openai/gpt-oss-20b)  (object, chat.completion)   
2  (model, openai/gpt-oss-20b)  (object, chat.completion)   

                       5                          6   \
0  (mcp_list_tools, None)  (service_tier, on_demand)   
1  (mcp_list_tools, None)  (service_tier, on_demand)   
2  (mcp_list_tools, None)  (service_tier, on_